# Renewable Energy Integration & Forecasting - Synthetic Data Generator

## Overview
This notebook generates synthetic data for renewable energy forecasting:
- **Solar generation** - PV output with irradiance and weather correlation
- **Wind generation** - Turbine output with wind speed patterns
- **Weather data** - Temperature, cloud cover, precipitation
- **Grid demand** - Load patterns and curtailment events
- **Market prices** - Real-time and day-ahead pricing

## Tables Created
| Table | Description | Volume |
|-------|-------------|--------|
| `dim_solar_plants` | Solar farm registry | ~20 rows |
| `dim_wind_farms` | Wind farm registry | ~15 rows |
| `fact_solar_generation` | Solar output readings | 100K+ rows/day |
| `fact_wind_generation` | Wind output readings | 50K+ rows/day |
| `fact_weather_forecast` | NWP model forecasts | 10K+ rows/day |
| `fact_market_prices` | Energy market prices | ~1K rows/day |

## How to Run in Fabric
1. Attach this notebook to a Lakehouse
2. Run all cells sequentially
3. Data will be written as Delta tables

In [ ]:
!pip install faker

In [ ]:
# Configuration
SEED = 42
NUM_SOLAR_PLANTS = 20
NUM_WIND_FARMS = 15

from datetime import datetime, timedelta
START_DATE = datetime(2024, 6, 1)  # Summer for good solar
END_DATE = datetime(2024, 6, 8)

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

np.random.seed(SEED)
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

print("Libraries loaded successfully")

## 1. Generate Solar Plant Dimension

In [ ]:
def generate_solar_plants(num_plants):
    plants = []
    
    for i in range(num_plants):
        capacity_mw = random.choice([10, 25, 50, 100, 150, 200])
        
        plants.append({
            'plant_id': f'SOL-{str(i+1).zfill(3)}',
            'plant_name': f'{fake.city()} Solar Farm',
            'latitude': round(random.uniform(32, 42), 4),  # US Southwest
            'longitude': round(random.uniform(-120, -95), 4),
            'capacity_mw': capacity_mw,
            'panel_type': random.choice(['Monocrystalline', 'Polycrystalline', 'Thin-Film']),
            'tracking_type': random.choice(['Fixed', 'Single-Axis', 'Dual-Axis']),
            'num_inverters': capacity_mw // 5 + 1,
            'install_date': fake.date_between(start_date='-10y', end_date='-1y'),
            'degradation_rate_pct': round(random.uniform(0.3, 0.8), 2),
            'grid_connection_point': f'SUB-{random.randint(100, 999)}'
        })
    
    return pd.DataFrame(plants)

df_solar_plants = generate_solar_plants(NUM_SOLAR_PLANTS)
print(f"Generated {len(df_solar_plants)} solar plants")
print(f"Total capacity: {df_solar_plants['capacity_mw'].sum()} MW")
df_solar_plants.head()

In [ ]:
def generate_wind_farms(num_farms):
    farms = []
    
    for i in range(num_farms):
        num_turbines = random.choice([20, 30, 50, 75, 100])
        turbine_capacity = random.choice([2.0, 2.5, 3.0, 4.0, 5.0])  # MW each
        
        farms.append({
            'farm_id': f'WND-{str(i+1).zfill(3)}',
            'farm_name': f'{fake.last_name()} Wind Farm',
            'latitude': round(random.uniform(35, 45), 4),
            'longitude': round(random.uniform(-105, -85), 4),
            'num_turbines': num_turbines,
            'turbine_capacity_mw': turbine_capacity,
            'total_capacity_mw': num_turbines * turbine_capacity,
            'hub_height_m': random.choice([80, 90, 100, 120]),
            'rotor_diameter_m': random.choice([100, 110, 130, 150]),
            'cut_in_speed_ms': round(random.uniform(3, 4), 1),
            'cut_out_speed_ms': 25.0,
            'rated_speed_ms': round(random.uniform(11, 14), 1),
            'install_date': fake.date_between(start_date='-15y', end_date='-1y'),
            'grid_connection_point': f'SUB-{random.randint(100, 999)}'
        })
    
    return pd.DataFrame(farms)

df_wind_farms = generate_wind_farms(NUM_WIND_FARMS)
print(f"Generated {len(df_wind_farms)} wind farms")
print(f"Total capacity: {df_wind_farms['total_capacity_mw'].sum()} MW")
df_wind_farms.head()

## 2. Generate Weather Forecast Data

In [ ]:
def generate_weather_forecasts(solar_plants, wind_farms, start_date, days=7):
    """Generate NWP-style weather forecasts"""
    forecasts = []
    
    # Combine all locations
    locations = []
    for _, plant in solar_plants.iterrows():
        locations.append(('solar', plant['plant_id'], plant['latitude'], plant['longitude']))
    for _, farm in wind_farms.iterrows():
        locations.append(('wind', farm['farm_id'], farm['latitude'], farm['longitude']))
    
    # Generate hourly forecasts
    for loc_type, loc_id, lat, lon in locations:
        current_time = start_date
        
        while current_time < start_date + timedelta(days=days):
            hour = current_time.hour
            
            # Solar irradiance (peaks at noon)
            if 6 <= hour <= 20:
                solar_zenith = abs(hour - 13) / 7  # 0 at noon, 1 at sunrise/sunset
                clear_sky_ghi = 1000 * (1 - solar_zenith) * np.cos(solar_zenith * np.pi / 2)
                cloud_factor = random.uniform(0.3, 1.0)  # Clouds reduce irradiance
                ghi = max(0, clear_sky_ghi * cloud_factor + np.random.normal(0, 50))
            else:
                ghi = 0
                cloud_factor = random.uniform(0, 1)
            
            # Wind speed (diurnal pattern + randomness)
            base_wind = 8 + 3 * np.sin((hour - 6) * np.pi / 12)  # Higher in afternoon
            wind_speed = max(0, base_wind + np.random.normal(0, 3))
            wind_direction = random.uniform(0, 360)
            
            # Temperature
            temp = 25 + 10 * np.sin((hour - 6) * np.pi / 12) + np.random.normal(0, 2)
            
            forecasts.append({
                'forecast_id': str(uuid.uuid4()),
                'location_type': loc_type,
                'location_id': loc_id,
                'forecast_timestamp': current_time,
                'issued_at': current_time - timedelta(hours=6),
                'ghi_wm2': round(ghi, 1),
                'dni_wm2': round(ghi * 0.8, 1),
                'dhi_wm2': round(ghi * 0.2, 1),
                'wind_speed_ms': round(wind_speed, 1),
                'wind_direction_deg': round(wind_direction, 0),
                'temperature_c': round(temp, 1),
                'humidity_pct': round(random.uniform(30, 80), 0),
                'cloud_cover_pct': round((1 - cloud_factor) * 100, 0),
                'precipitation_mm': round(max(0, np.random.exponential(0.5) if cloud_factor < 0.5 else 0), 1)
            })
            
            current_time += timedelta(hours=1)
    
    return pd.DataFrame(forecasts)

df_weather = generate_weather_forecasts(df_solar_plants, df_wind_farms, START_DATE, days=7)
print(f"Generated {len(df_weather)} weather forecasts")
df_weather.head(10)

## 3. Generate Solar Generation Data

In [ ]:
def generate_solar_generation(solar_plants, weather_df, start_date, days=7):
    """Generate solar power output correlated with weather"""
    readings = []
    
    for _, plant in solar_plants.iterrows():
        plant_weather = weather_df[
            (weather_df['location_id'] == plant['plant_id']) & 
            (weather_df['location_type'] == 'solar')
        ].copy()
        
        # Calculate age-based degradation
        age_years = (datetime.now() - pd.to_datetime(plant['install_date'])).days / 365
        efficiency = 1 - (age_years * plant['degradation_rate_pct'] / 100)
        
        # Tracking multiplier
        tracking_bonus = {'Fixed': 1.0, 'Single-Axis': 1.25, 'Dual-Axis': 1.35}
        tracking_factor = tracking_bonus.get(plant['tracking_type'], 1.0)
        
        for _, weather in plant_weather.iterrows():
            # Power output based on irradiance and efficiency
            theoretical_output = plant['capacity_mw'] * (weather['ghi_wm2'] / 1000)
            actual_output = theoretical_output * efficiency * tracking_factor
            
            # Temperature derating (high temps reduce output)
            temp_derate = 1 - max(0, (weather['temperature_c'] - 25) * 0.004)
            actual_output *= temp_derate
            
            # Add some noise
            actual_output = max(0, actual_output + np.random.normal(0, actual_output * 0.02))
            
            # Curtailment (random grid constraints)
            is_curtailed = random.random() < 0.02
            if is_curtailed:
                actual_output *= random.uniform(0.3, 0.7)
            
            readings.append({
                'reading_id': str(uuid.uuid4()),
                'plant_id': plant['plant_id'],
                'timestamp': weather['forecast_timestamp'],
                'power_output_mw': round(actual_output, 2),
                'capacity_factor_pct': round((actual_output / plant['capacity_mw']) * 100, 1),
                'irradiance_wm2': weather['ghi_wm2'],
                'panel_temp_c': round(weather['temperature_c'] + 20, 1),  # Panels run hotter
                'is_curtailed': is_curtailed,
                'availability_pct': round(random.uniform(95, 100), 1),
                'inverter_efficiency_pct': round(random.uniform(96, 99), 1)
            })
    
    return pd.DataFrame(readings)

df_solar_gen = generate_solar_generation(df_solar_plants, df_weather, START_DATE)
print(f"Generated {len(df_solar_gen)} solar generation readings")
print(f"Peak output: {df_solar_gen['power_output_mw'].max():.1f} MW")
print(f"Average capacity factor: {df_solar_gen['capacity_factor_pct'].mean():.1f}%")
df_solar_gen.head()

## 4. Generate Wind Generation Data

In [ ]:
def wind_power_curve(wind_speed, cut_in, rated_speed, cut_out, capacity):
    """Calculate power output from wind speed using standard power curve"""
    if wind_speed < cut_in:
        return 0
    elif wind_speed < rated_speed:
        # Cubic relationship in partial load region
        return capacity * ((wind_speed - cut_in) / (rated_speed - cut_in)) ** 3
    elif wind_speed <= cut_out:
        return capacity
    else:
        return 0  # Cut-out for safety

def generate_wind_generation(wind_farms, weather_df, start_date, days=7):
    """Generate wind power output correlated with weather"""
    readings = []
    
    for _, farm in wind_farms.iterrows():
        farm_weather = weather_df[
            (weather_df['location_id'] == farm['farm_id']) & 
            (weather_df['location_type'] == 'wind')
        ].copy()
        
        for _, weather in farm_weather.iterrows():
            # Calculate power from wind speed
            power = wind_power_curve(
                weather['wind_speed_ms'],
                farm['cut_in_speed_ms'],
                farm['rated_speed_ms'],
                farm['cut_out_speed_ms'],
                farm['total_capacity_mw']
            )
            
            # Turbine availability (some offline for maintenance)
            availability = random.uniform(0.92, 1.0)
            power *= availability
            
            # Add noise
            power = max(0, power + np.random.normal(0, power * 0.03))
            
            # Curtailment
            is_curtailed = random.random() < 0.03
            if is_curtailed:
                power *= random.uniform(0.2, 0.6)
            
            readings.append({
                'reading_id': str(uuid.uuid4()),
                'farm_id': farm['farm_id'],
                'timestamp': weather['forecast_timestamp'],
                'power_output_mw': round(power, 2),
                'capacity_factor_pct': round((power / farm['total_capacity_mw']) * 100, 1),
                'wind_speed_ms': weather['wind_speed_ms'],
                'wind_direction_deg': weather['wind_direction_deg'],
                'is_curtailed': is_curtailed,
                'turbines_online': int(farm['num_turbines'] * availability),
                'turbines_total': farm['num_turbines']
            })
    
    return pd.DataFrame(readings)

df_wind_gen = generate_wind_generation(df_wind_farms, df_weather, START_DATE)
print(f"Generated {len(df_wind_gen)} wind generation readings")
print(f"Peak output: {df_wind_gen['power_output_mw'].max():.1f} MW")
print(f"Average capacity factor: {df_wind_gen['capacity_factor_pct'].mean():.1f}%")
df_wind_gen.head()

## 5. Generate Market Price Data

In [ ]:
def generate_market_prices(start_date, days=7):
    """Generate energy market prices (LMP style)"""
    prices = []
    
    nodes = ['NODE_NORTH', 'NODE_SOUTH', 'NODE_EAST', 'NODE_WEST', 'NODE_HUB']
    
    current_time = start_date
    while current_time < start_date + timedelta(days=days):
        hour = current_time.hour
        is_weekend = current_time.weekday() >= 5
        
        # Base price with time-of-day pattern
        if is_weekend:
            base_price = 30 + 15 * np.sin((hour - 6) * np.pi / 12)
        else:
            # Higher prices during business hours
            if 7 <= hour <= 20:
                base_price = 45 + 25 * np.sin((hour - 6) * np.pi / 12)
            else:
                base_price = 25
        
        for node in nodes:
            # Node-specific variation
            node_factor = random.uniform(0.9, 1.1)
            lmp = base_price * node_factor + np.random.normal(0, 5)
            
            # Occasional price spikes
            if random.random() < 0.02:
                lmp *= random.uniform(2, 5)
            
            # Occasional negative prices (excess renewables)
            if random.random() < 0.03 and 10 <= hour <= 15:
                lmp = -random.uniform(5, 30)
            
            prices.append({
                'price_id': str(uuid.uuid4()),
                'node_id': node,
                'timestamp': current_time,
                'lmp_usd_mwh': round(lmp, 2),
                'energy_component': round(lmp * 0.85, 2),
                'congestion_component': round(lmp * 0.10, 2),
                'losses_component': round(lmp * 0.05, 2),
                'market_type': 'Real-Time'
            })
        
        current_time += timedelta(hours=1)
    
    return pd.DataFrame(prices)

df_prices = generate_market_prices(START_DATE, days=7)
print(f"Generated {len(df_prices)} price records")
print(f"Price range: ${df_prices['lmp_usd_mwh'].min():.2f} to ${df_prices['lmp_usd_mwh'].max():.2f}/MWh")
print(f"Negative price hours: {len(df_prices[df_prices['lmp_usd_mwh'] < 0])}")
df_prices.head()

## 6. Visualize Generation Patterns

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Solar generation over time
solar_hourly = df_solar_gen.groupby('timestamp')['power_output_mw'].sum()
axes[0].plot(solar_hourly.index, solar_hourly.values, color='orange', linewidth=1)
axes[0].fill_between(solar_hourly.index, solar_hourly.values, alpha=0.3, color='orange')
axes[0].set_title('Total Solar Generation')
axes[0].set_ylabel('MW')
axes[0].grid(True, alpha=0.3)

# Wind generation over time
wind_hourly = df_wind_gen.groupby('timestamp')['power_output_mw'].sum()
axes[1].plot(wind_hourly.index, wind_hourly.values, color='blue', linewidth=1)
axes[1].fill_between(wind_hourly.index, wind_hourly.values, alpha=0.3, color='blue')
axes[1].set_title('Total Wind Generation')
axes[1].set_ylabel('MW')
axes[1].grid(True, alpha=0.3)

# Market prices
hub_prices = df_prices[df_prices['node_id'] == 'NODE_HUB']
axes[2].plot(hub_prices['timestamp'], hub_prices['lmp_usd_mwh'], color='green', linewidth=1)
axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('Hub Market Prices')
axes[2].set_ylabel('$/MWh')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Data Validation

In [ ]:
print("=== Data Validation Report ===")
print(f"\nSolar Plants: {len(df_solar_plants)} records")
print(f"  - Total capacity: {df_solar_plants['capacity_mw'].sum()} MW")

print(f"\nWind Farms: {len(df_wind_farms)} records")
print(f"  - Total capacity: {df_wind_farms['total_capacity_mw'].sum()} MW")

print(f"\nWeather Forecasts: {len(df_weather)} records")
print(f"  - Solar locations: {len(df_weather[df_weather['location_type'] == 'solar']['location_id'].unique())}")
print(f"  - Wind locations: {len(df_weather[df_weather['location_type'] == 'wind']['location_id'].unique())}")

print(f"\nSolar Generation: {len(df_solar_gen)} records")
print(f"  - Total energy: {df_solar_gen['power_output_mw'].sum():,.0f} MWh")
print(f"  - Curtailed hours: {df_solar_gen['is_curtailed'].sum()}")

print(f"\nWind Generation: {len(df_wind_gen)} records")
print(f"  - Total energy: {df_wind_gen['power_output_mw'].sum():,.0f} MWh")
print(f"  - Curtailed hours: {df_wind_gen['is_curtailed'].sum()}")

print(f"\nMarket Prices: {len(df_prices)} records")
print(f"  - Avg price: ${df_prices['lmp_usd_mwh'].mean():.2f}/MWh")

## 8. Write to Lakehouse

In [ ]:
# Uncomment for Fabric execution
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()

# spark.createDataFrame(df_solar_plants).write.mode("overwrite").format("delta").saveAsTable("dim_solar_plants")
# spark.createDataFrame(df_wind_farms).write.mode("overwrite").format("delta").saveAsTable("dim_wind_farms")
# spark.createDataFrame(df_weather).write.mode("append").format("delta").saveAsTable("fact_weather_forecast")
# spark.createDataFrame(df_solar_gen).write.mode("append").format("delta").saveAsTable("fact_solar_generation")
# spark.createDataFrame(df_wind_gen).write.mode("append").format("delta").saveAsTable("fact_wind_generation")
# spark.createDataFrame(df_prices).write.mode("append").format("delta").saveAsTable("fact_market_prices")

# Local testing
df_solar_plants.to_csv('dim_solar_plants.csv', index=False)
df_solar_gen.to_csv('fact_solar_generation.csv', index=False)
print("Data saved successfully")

## Known Limitations
1. Weather correlations are simplified - real NWP models have complex spatial patterns
2. Wind power curve is idealized - actual curves vary by manufacturer
3. Market prices don't reflect actual grid constraints

## Next Steps
1. Train forecasting model on historical generation vs. weather
2. Build ensemble forecasts combining multiple NWP sources
3. Optimize curtailment vs. storage charging decisions